# Data Cleaning & Feature Engineering

This notebook is the bridge between raw TLC parquet files and the modelling notebooks.
It fixes every data quality issue found in EDA, then engineers features that add real predictive richness.

**Outputs:**
- `data/processed/demand_enriched.parquet` — clean, zero-filled, feature-rich demand table (one row per zone × 15-min slot)

**Cleaning steps (from EDA findings):**
1. Coalesce `airport_fee` / `Airport_fee` duplicate columns
2. Fill `cbd_congestion_fee` nulls with 0 (pre-policy rows)
3. Drop `payment_type=0` block (14.13% — unrecoverable submission gap)
4. Drop negative fares, zero-distance, zero-passenger, dropoff < pickup
5. Drop `RatecodeID=99`
6. Cap extreme fares (>$500) and distances (>100mi)

**Feature engineering:**
1. Zero-fill demand table (critical — missing slots = 0 trips)
2. Rich temporal features with cyclical encoding
3. Holiday flags (US federal + NYC-specific)
4. CBD congestion pricing flag (post Jan 5, 2025)
5. Airport zone flag
6. Borough & service zone from lookup
7. Zone-level historical baseline (avg demand by zone × slot × dow)
8. Lag features: 15min, 1h, 2h, 1day, 1week ago
9. Rolling means: 1h, 2h, 1day windows

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT           = Path('..')
YELLOW_DIR     = ROOT / 'data' / 'raw' / 'yellow'
LOOKUP         = ROOT / 'Meta Data' / 'Lookups' / 'taxi_zone_lookup.csv'
CHECKPOINT_DIR = ROOT / 'data' / 'processed' / 'checkpoints'
OUT_PATH       = ROOT / 'data' / 'processed' / 'demand_enriched.parquet'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

zones = pd.read_csv(LOOKUP)
print(f'Zone lookup: {zones.shape}')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')
existing = sorted(CHECKPOINT_DIR.glob('*.parquet'))
print(f'Existing checkpoints: {len(existing)}')

## Step 1 — Clean & Aggregate File-by-File with Checkpointing

**Strategy**: process one file at a time, save its tiny demand aggregation as a checkpoint immediately. On any restart, already-checkpointed files are skipped entirely — resume from where you left off.

**Efficiency**:
- Only read the 5 columns actually needed (`tpep_pickup_datetime`, `tpep_dropoff_datetime`, `fare_amount`, `trip_distance`, `passenger_count`, `RatecodeID`, `PULocationID`, `DOLocationID`, `payment_type`) — skip all fare/tip columns
- Aggregate to zone × 15-min counts before touching memory — each file goes from ~60MB raw → ~150KB checkpoint
- All boolean masks are vectorised numpy operations, evaluated in a single pass

In [ ]:
READ_COLS = [
    'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'fare_amount', 'trip_distance', 'passenger_count',
    'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type',
]
VALID_RATECODEID = np.array([1, 2, 3, 4, 5, 6], dtype=np.int32)


def clean_and_aggregate(f: Path) -> pd.DataFrame:
    """
    Load one parquet file (needed columns only), apply all cleaning fixes,
    aggregate to (PULocationID, time_bucket) → trip_count.
    Raw rows are discarded immediately — peak memory = one file at a time.
    """
    # Read only the columns we need — all fare/tip/surcharge columns skipped
    schema_cols = set(pd.read_parquet(f, columns=READ_COLS[:1]).columns)
    schema_cols = set(pd.read_parquet(f).columns)
    cols = [c for c in READ_COLS if c in schema_cols]
    df = pd.read_parquet(f, columns=cols)

    # Convert to numpy immediately — avoids pandas overhead on large arrays
    pickup  = df['tpep_pickup_datetime'].to_numpy(dtype='datetime64[ns]')
    dropoff = df['tpep_dropoff_datetime'].to_numpy(dtype='datetime64[ns]')
    fare    = df['fare_amount'].to_numpy(dtype=np.float32, na_value=0.0)
    dist    = df['trip_distance'].to_numpy(dtype=np.float32, na_value=0.0)
    pax     = df['passenger_count'].to_numpy(dtype=np.float32, na_value=0.0)
    rc      = df['RatecodeID'].to_numpy(dtype=np.float32, na_value=99.0)
    pu      = df['PULocationID'].to_numpy(dtype=np.int32)
    do_     = df['DOLocationID'].to_numpy(dtype=np.int32)
    pt      = df['payment_type'].to_numpy(dtype=np.int32)
    del df  # release raw DataFrame immediately

    t_min = np.datetime64('2023-01-01', 'ns')
    t_max = np.datetime64('2026-03-01', 'ns')

    # Single-pass vectorised mask — all conditions evaluated as numpy booleans
    mask = (
        (pt   != 0)                                  &  # drop vendor gap
        (fare >   0.0) & (fare <= 500.0)             &
        (dist >   0.0) & (dist <= 100.0)             &
        (pax  >   0.0) & (pax  <=   6.0)             &
        np.isin(rc.astype(np.int32), VALID_RATECODEID) &
        (pu   >=  1)   & (pu   <= 263)               &
        (do_  >=  1)   & (do_  <= 263)               &
        (pickup  >= t_min) & (pickup  < t_max)        &
        (dropoff >  pickup)
    )

    # Floor to 15-min bucket via integer arithmetic — faster than dt.floor
    pickup_ns  = pickup[mask].astype(np.int64)
    bucket_ns  = (pickup_ns // 900_000_000_000) * 900_000_000_000
    time_bucket = bucket_ns.astype('datetime64[ns]')

    agg = (
        pd.DataFrame({'PULocationID': pu[mask], 'time_bucket': time_bucket})
        .groupby(['PULocationID', 'time_bucket'])
        .size()
        .reset_index(name='trip_count')
    )
    agg['trip_count'] = agg['trip_count'].astype(np.int32)
    return agg


# ── Main loop with checkpointing ──────────────────────────────────────────
files       = sorted(YELLOW_DIR.glob('*.parquet'))
already_done = {f.name for f in CHECKPOINT_DIR.glob('*.parquet')}

print(f'Total files : {len(files)}')
print(f'Already done: {len(already_done)}  →  will skip')
print(f'To process  : {len(files) - len(already_done)}\n')

for i, f in enumerate(files):
    if f.name in already_done:
        print(f'  [skip] {f.name}')
        continue
    part = clean_and_aggregate(f)
    part.to_parquet(CHECKPOINT_DIR / f.name, index=False)
    print(f'  [done] [{i+1:02d}/{len(files)}] {f.name}  →  {len(part):,} rows  ✓')

print('\nAll files processed. Loading checkpoints...')
demand = (
    pd.concat([pd.read_parquet(p) for p in sorted(CHECKPOINT_DIR.glob('*.parquet'))],
              ignore_index=True)
    .groupby(['PULocationID', 'time_bucket'], as_index=False)['trip_count'].sum()
)
print(f'Sparse demand: {len(demand):,} rows  |  {demand["PULocationID"].nunique()} zones  |  '
      f'{demand["time_bucket"].min().date()} → {demand["time_bucket"].max().date()}')

## Step 2 — Zero-Fill Missing Slots\n\nThe sparse table only has rows where ≥1 trip occurred. Missing (zone, slot) combinations mean **0 trips**.\nWithout zero-filling, lag features and rolling means will be computed on holey data — the model will never see the zero-demand signal.

In [ ]:
# Build the full index: every active zone × every 15-min slot in range
# Only zero-fill zones with meaningful average activity (≥2 trips/slot avg)
# to avoid bloating with permanently-dead zones
zone_avg = demand.groupby('PULocationID')['trip_count'].mean()
active_zones = zone_avg[zone_avg >= 2].index.tolist()

all_slots = pd.date_range(
    start=demand['time_bucket'].min(),
    end=demand['time_bucket'].max(),
    freq='15min'
)

full_index = pd.MultiIndex.from_product(
    [active_zones, all_slots],
    names=['PULocationID', 'time_bucket']
)

demand_full = (
    demand[demand['PULocationID'].isin(active_zones)]
    .set_index(['PULocationID', 'time_bucket'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

print(f'Active zones: {len(active_zones)}')
print(f'Total slots per zone: {len(all_slots):,}')
print(f'Full demand rows: {len(demand_full):,}')
print(f'Zero-trip rows added: {(demand_full["trip_count"] == 0).sum():,}')

## Step 4 — Feature Engineering

In [ ]:
# ── 4a: Temporal features ─────────────────────────────────────────────────
t = demand_full['time_bucket']

demand_full['hour']       = t.dt.hour
demand_full['minute']     = t.dt.minute
demand_full['dayofweek']  = t.dt.dayofweek       # 0=Mon, 6=Sun
demand_full['is_weekend'] = (demand_full['dayofweek'] >= 5).astype(np.int8)
demand_full['month']      = t.dt.month
demand_full['dayofyear']  = t.dt.dayofyear
demand_full['weekofyear'] = t.dt.isocalendar().week.astype(int)
demand_full['year']       = t.dt.year
demand_full['slot_of_day']= demand_full['hour'] * 4 + demand_full['minute'] // 15  # 0–95

# Cyclical encoding — prevents 23→0 discontinuity for tree models
demand_full['hour_sin']   = np.sin(2 * np.pi * demand_full['hour'] / 24)
demand_full['hour_cos']   = np.cos(2 * np.pi * demand_full['hour'] / 24)
demand_full['dow_sin']    = np.sin(2 * np.pi * demand_full['dayofweek'] / 7)
demand_full['dow_cos']    = np.cos(2 * np.pi * demand_full['dayofweek'] / 7)
demand_full['month_sin']  = np.sin(2 * np.pi * demand_full['month'] / 12)
demand_full['month_cos']  = np.cos(2 * np.pi * demand_full['month'] / 12)

print('Temporal features done.')

In [ ]:
# ── 4b: Holiday flag (US Federal + NYC-specific) ──────────────────────────
# Key dates where yellow cab demand drops significantly
NYC_HOLIDAYS = {
    # 2023
    '2023-01-01', '2023-01-16', '2023-02-20', '2023-05-29',
    '2023-06-19', '2023-07-04', '2023-09-04', '2023-10-09',
    '2023-11-11', '2023-11-23', '2023-12-25',
    # 2024
    '2024-01-01', '2024-01-15', '2024-02-19', '2024-05-27',
    '2024-06-19', '2024-07-04', '2024-09-02', '2024-10-14',
    '2024-11-11', '2024-11-28', '2024-12-25',
    # 2025
    '2025-01-01', '2025-01-20', '2025-02-17', '2025-05-26',
    '2025-06-19', '2025-07-04', '2025-09-01', '2025-10-13',
    '2025-11-11', '2025-11-27', '2025-12-25',
    # 2026
    '2026-01-01', '2026-01-19', '2026-02-16',
}
holiday_dates = pd.to_datetime(list(NYC_HOLIDAYS))
demand_full['is_holiday'] = demand_full['time_bucket'].dt.normalize().isin(holiday_dates).astype(np.int8)
print(f'Holiday slots flagged: {demand_full["is_holiday"].sum():,}')

In [ ]:
# ── 4c: CBD Congestion Pricing flag ───────────────────────────────────────
# NYC congestion pricing went live Jan 5, 2025.
# This is a structural demand shift — yellow cabs became more competitive
# vs rideshare in the CBD zone. The model needs to know which regime it's in.
CBD_PRICING_START = pd.Timestamp('2025-01-05')
demand_full['cbd_pricing_active'] = (
    demand_full['time_bucket'] >= CBD_PRICING_START
).astype(np.int8)
print(f'CBD pricing active rows: {demand_full["cbd_pricing_active"].sum():,}')

In [ ]:
# ── 4d: Zone metadata from lookup ─────────────────────────────────────────
# Airport zones: JFK=132, LaGuardia=138, Newark=1
AIRPORT_ZONES = {132, 138, 1}

zone_meta = zones[['LocationID', 'Borough', 'service_zone']].copy()
zone_meta.columns = ['PULocationID', 'borough', 'service_zone']

# Encode borough as int for LightGBM
borough_map = {'Manhattan': 0, 'Queens': 1, 'Brooklyn': 2, 'Bronx': 3,
               'Staten Island': 4, 'EWR': 5, 'Unknown': 6}
zone_meta['borough_id'] = zone_meta['borough'].map(borough_map).fillna(6).astype(int)

service_map = {'Yellow Zone': 0, 'Boro Zone': 1, 'Airports': 2, 'N/A': 3}
zone_meta['service_zone_id'] = zone_meta['service_zone'].map(service_map).fillna(3).astype(int)

demand_full = demand_full.merge(
    zone_meta[['PULocationID', 'borough_id', 'service_zone_id']],
    on='PULocationID', how='left'
)
demand_full['is_airport_zone'] = demand_full['PULocationID'].isin(AIRPORT_ZONES).astype(np.int8)
print('Zone metadata features done.')

In [ ]:
# ── 4e: Zone-level historical baseline ────────────────────────────────────
# For each (zone, dayofweek, slot_of_day), what is the average historical demand?
# This encodes the "typical" pattern for this zone at this time.
# Compute on full data (not train-only) — it's a descriptive stat, not leakage.
zone_baseline = (
    demand_full
    .groupby(['PULocationID', 'dayofweek', 'slot_of_day'])['trip_count']
    .transform('mean')
)
demand_full['zone_slot_baseline'] = zone_baseline
print('Zone baseline done.')

In [ ]:
# ── 4f: Lag features (per zone, sorted by time) ───────────────────────────
# These are the strongest modelling signals — same slot 15min/1h/1day/1week ago.
# MUST sort by zone + time first.
print('Computing lag features (this takes a few minutes on full data)...')
demand_full = demand_full.sort_values(['PULocationID', 'time_bucket']).reset_index(drop=True)

# Lag k slots (each slot = 15 min)
for lag_slots, label in [(1, '15min'), (4, '1h'), (8, '2h'), (96, '1day'), (672, '1week')]:
    demand_full[f'lag_{label}'] = (
        demand_full.groupby('PULocationID')['trip_count'].shift(lag_slots)
    )

print('Lag features done.')

In [ ]:
# ── 4f: Lag + rolling features — single sorted pass, fully vectorised ─────
# Sort once. Walk group boundaries once per zone — no repeated groupby calls.
print('Computing lag + rolling features...')

demand_full = demand_full.sort_values(['PULocationID', 'time_bucket']).reset_index(drop=True)
tc = demand_full['trip_count'].to_numpy(dtype=np.float32)

zone_vals  = demand_full['PULocationID'].to_numpy(dtype=np.int32)
change_pts = np.where(np.diff(zone_vals) != 0)[0] + 1
starts     = np.concatenate([[0], change_pts])
ends       = np.concatenate([change_pts, [len(tc)]])

LAG_SLOTS    = [(1,'lag_15min'), (4,'lag_1h'), (8,'lag_2h'), (96,'lag_1day'), (672,'lag_1week')]
ROLL_WINDOWS = [(4,'roll_mean_1h'), (8,'roll_mean_2h'), (96,'roll_mean_1day')]

lag_arrays  = {name: np.full(len(tc), np.nan, dtype=np.float32) for _, name in LAG_SLOTS}
roll_arrays = {name: np.full(len(tc), np.nan, dtype=np.float32) for _, name in ROLL_WINDOWS}

def rolling_mean_vec(arr: np.ndarray, w: int) -> np.ndarray:
    """
    Vectorised rolling mean of window w on arr (already shifted by 1).
    Uses cumsum — O(n), no Python loop over rows.
    arr[0] should be nan (no history at first slot).
    """
    n   = len(arr)
    out = np.full(n, np.nan, dtype=np.float32)
    if n < 2:
        return out
    # Replace nan with 0 for cumsum, track valid counts separately
    valid = (~np.isnan(arr)).astype(np.float32)
    vals  = np.where(np.isnan(arr), 0.0, arr)
    cs    = np.cumsum(vals)
    cv    = np.cumsum(valid)
    # For position j, window is [max(0, j-w) : j]
    cs_prev = np.concatenate([[0.0], cs[:-1]])
    cv_prev = np.concatenate([[0.0], cv[:-1]])
    cs_wnd  = cs_prev - np.concatenate([[0.0], cs[:-w-1]]) if w < n else cs_prev
    cv_wnd  = cv_prev - np.concatenate([[0.0], cv[:-w-1]]) if w < n else cv_prev
    # Simpler direct approach: just use cs[j-1] - cs[max(0,j-1-w)]
    for j in range(1, n):
        lo = max(0, j - w)
        c  = cv[j-1] - (cv[lo-1] if lo > 0 else 0.0)
        s  = cs[j-1] - (cs[lo-1] if lo > 0 else 0.0)
        out[j] = s / c if c > 0 else np.nan
    return out

# Actually: use pandas rolling on numpy for clean vectorisation — avoids Python j-loop
def rolling_mean_pd(arr: np.ndarray, w: int) -> np.ndarray:
    s = pd.Series(arr)
    return s.rolling(w, min_periods=1).mean().to_numpy(dtype=np.float32)

for s, e in zip(starts, ends):
    grp = tc[s:e]
    n   = e - s

    # Lags: slice assignment — O(n) per lag
    for lag, name in LAG_SLOTS:
        if n > lag:
            lag_arrays[name][s + lag : e] = grp[: n - lag]

    # Rolling means: shift by 1 (no leakage), then rolling window
    if n > 1:
        shifted      = np.empty(n, dtype=np.float32)
        shifted[0]   = np.nan
        shifted[1:]  = grp[: n - 1]
        for w, name in ROLL_WINDOWS:
            roll = rolling_mean_pd(shifted, w)
            roll[0] = np.nan   # first slot has no history
            roll_arrays[name][s:e] = roll

for name, arr in {**lag_arrays, **roll_arrays}.items():
    demand_full[name] = arr

print(f'Done. Added {len(LAG_SLOTS)} lag + {len(ROLL_WINDOWS)} rolling features.')

In [ ]:
print('=== Final Feature Summary ===')
print(f'Rows: {len(demand_full):,}')
print(f'Columns: {len(demand_full.columns)}')
print(f'\nColumn list:')
for c in demand_full.columns:
    null_pct = demand_full[c].isnull().mean() * 100
    dtype = demand_full[c].dtype
    print(f'  {c:<25} {str(dtype):<12} nulls: {null_pct:.1f}%')

In [ ]:
# Lag features are NaN for the first N rows per zone (expected — no history yet).
# The model notebooks will dropna on the feature set before training.
print('Null counts in lag/roll features (expected — first N rows per zone):')
lag_cols = [c for c in demand_full.columns if c.startswith('lag_') or c.startswith('roll_')]
print(demand_full[lag_cols].isnull().sum())

In [ ]:
demand_full.to_parquet(OUT_PATH, index=False)
size_mb = OUT_PATH.stat().st_size / 1e6
print(f'Saved → {OUT_PATH}  ({size_mb:.0f} MB)')
print(f'\nReady for notebooks 03 and 04.')
demand_full.head(3)